# Pandas: scaling to large datasets

The goal of this notebook is to learn several practices for manipulating large datasets with pandas.

In [2]:
import random
import string
import numpy as np
import pandas as pd
from datetime import datetime
import pathlib
%load_ext memory_profiler

The memory_profiler extension is already loaded. To reload it, use:
  %reload_ext memory_profiler


Create a large dataset

In [3]:
%%time
def gen_random_string(length:int=32) -> str:
    return ''.join(random.choices(string.ascii_uppercase + string.digits, k=length))
    
def make_timeseries(start="2000-01-01", end="2000-12-31", freq="1D", seed=None):

    index = pd.date_range(start=start, end=end, freq=freq, name="timestamp")
    n = len(index)
    np.random.seed = seed
    columns = {
        'cat': np.random.choice(['cat1','cat2','cat3','cat4','cat5'],n),
        'str1':[gen_random_string() for _ in range(n)],
        'str2':[gen_random_string() for _ in range(n)],
        'a': np.random.rand(n),
        'b': np.random.rand(n),
        'c': np.random.randint(1,100,n),
    }

    df = pd.DataFrame(columns, index=index, columns=sorted(columns))
    if df.index[-1] == end:
        df = df.iloc[:-1]
    return df

timeseries = [
    make_timeseries(start=datetime(2020,1,1), end=datetime(2023,12,31), freq='1min', seed=10).rename(columns=lambda x: f"{x}_{i}")
    for i in range(5)
]
df = pd.concat(timeseries, axis=1)

CPU times: user 34.3 s, sys: 1.44 s, total: 35.8 s
Wall time: 35.8 s


Print the first rows to see what the data looks like.

In [4]:
df.head()

,a_0,b_0,c_0,cat_0,str1_0,str2_0,a_1,b_1,c_1,cat_1,...,c_3,cat_3,str1_3,str2_3,a_4,b_4,c_4,cat_4,str1_4,str2_4
timestamp,,,,,,,,,,,,,,,,,,,,,
2020-01-01 00:00:00,0.703083,0.568984,78,cat2,BYYJ7Z960GUIPORHM9C302I747YG7DEL,BEGHD76MZBXAOQUK9IJ825KOD0Y2PUST,0.883547,0.417814,45,cat3,...,68,cat1,L0WZD1BIO6RH0V9DIKKLLBBAU97X11MC,H0FNVYIGQRF4RA8U4NGL4QA1ADA3TI4S,0.415569,0.639620,93,cat5,NAXNYLINYTRMAPI2YYXK3FPBMR904ZEF,GXE5S99QHX62IIN82BS9T9RQYOZZ2E44
2020-01-01 00:01:00,0.709437,0.753189,49,cat3,VPAKHHCHCF1GNQY989CGBEJWS2K6ZLYH,YJZFHO0194094MPTRI6EE0SQYDF1A05V,0.849775,0.644855,54,cat5,...,59,cat2,R1X905NBOGOSC8A32VRQLHNCDGGQYPJD,63KR648639E6VVABV10A0NTY4GCUH90D,0.176228,0.162556,70,cat4,I6TA99J2N2HYP82JKUK6M2COVB3HK1HE,4F5O8LGDPZR28FGZZ7WWYN3VBC9BFWCL
2020-01-01 00:02:00,0.283528,0.368110,89,cat2,HYJS2ZMEZ5HVSJVKF25P3DM4FZCNGC1O,OBEZAJOFS5AKJ4U3914FXW86NNBHAHZC,0.758266,0.567998,85,cat3,...,1,cat4,2SRY5UB8FJ7K942SZPEHBJNT6IEUO75S,KV7Q0M6NDBXLWUOMPAJL9S33CC2MI086,0.123520,0.711349,91,cat4,SUHESP6MWJLDLR9M3E5F6V7IIDDHHM1J,AVA6FC72V0JGQK2J1E9ON1N4YQMK3DWK
2020-01-01 00:03:00,0.974066,0.820260,6,cat4,U11DHD90WA52AATBY62IZSU0YVCVA74W,4Z4TCV9WBNKKQ9QRCMQU7L0UJ40ULV6L,0.668060,0.231752,25,cat4,...,43,cat3,169PGIRI6P9S02ICKODMNBXADIG9SSS5,C8KS8VC8LBHS2HG89YSYL941BCVABVB9,0.678583,0.520550,42,cat1,ADS05EXJ5CD7UBLPZHV1JF8ETNKOWF0X,TNMNCGPJX4SQGB5K92L3H1RTZMEYJ4XI
2020-01-01 00:04:00,0.633060,0.565444,89,cat3,J9R9NDOFM9F12RZD4C0WE2DTBHMDMANI,4NI71D1RYHFHZIR32CGUTG3EVQBKZ0VX,0.337518,0.691829,29,cat2,...,65,cat1,IU8LRX3IULGJ79OF0FW73CCTQS6CMAH7,10UXT9F71V9W0L0XDNUMCAYAF9GCSG6N,0.103360,0.747805,84,cat3,PY1R00J5MCJPKE8XBORTJLLW2BC37PKV,UX8ZE3KMYO1PRRQKTLWIDM8SQ3N7J3MQ


The method `info(memory_usage='deep')` returns the column types and also gives the memory usage of the dataframe.

In [5]:
df.info(memory_usage='deep')

<class 'pandas.DataFrame'>
DatetimeIndex: 2102400 entries, 2020-01-01 00:00:00 to 2023-12-30 23:59:00
Freq: min
Data columns (total 30 columns):
 #   Column  Dtype  
---  ------  -----  
 0   a_0     float64
 1   b_0     float64
 2   c_0     int64  
 3   cat_0   str    
 4   str1_0  str    
 5   str2_0  str    
 6   a_1     float64
 7   b_1     float64
 8   c_1     int64  
 9   cat_1   str    
 10  str1_1  str    
 11  str2_1  str    
 12  a_2     float64
 13  b_2     float64
 14  c_2     int64  
 15  cat_2   str    
 16  str1_2  str    
 17  str2_2  str    
 18  a_3     float64
 19  b_3     float64
 20  c_3     int64  
 21  cat_3   str    
 22  str1_3  str    
 23  str2_3  str    
 24  a_4     float64
 25  b_4     float64
 26  c_4     int64  
 27  cat_4   str    
 28  str1_4  str    
 29  str2_4  str    
dtypes: float64(10), int64(5), str(15)
memory usage: 1.2 GB


Write the dataframe 

In [6]:
pathlib.Path("data").mkdir(parents=True,exist_ok=True)
df.to_parquet("timeseries.parquet")

## Load only useful data

Imagine you're only interested in a subset of the dataset's columns `['a_0','a_1','cat_0','str1_0','str1_1']`. Then there are two ways to proceed: 
 * either load the entire dataset and then filter out the columns you're interested in
 * or read only the columns you're interested in

Compare the two loading methods. You can use the magic command `%time` and `%memit` to compare the time and the memory usage of the two calls.

Look at the `read_parquet`method

In [7]:
?pd.read_parquet

Signature:
pd.read_parquet(
    path: 'FilePath | ReadBuffer[bytes]',
    engine: 'str' = 'auto',
    columns: 'list[str] | None' = None,
    storage_options: 'StorageOptions | None' = None,
    dtype_backend: 'DtypeBackend | lib.NoDefault' = <no_default>,
    filesystem: 'Any' = None,
    filters: 'list[tuple] | list[list[tuple]] | None' = None,
    to_pandas_kwargs: 'dict | None' = None,
    **kwargs,
) -> 'DataFrame'
Docstring:
Load a parquet object from the file path, returning a DataFrame.

The function automatically handles reading the data from a parquet file
and creates a DataFrame with the appropriate structure.

Parameters
----------
path : str, path object or file-like object
    String, path object (implementing ``os.PathLike[str]``), or file-like
    object implementing a binary ``read()`` function.
    The string could be a URL. Valid URL schemes include http, ftp, s3,
    gs, and file. For file URLs, a host is expected. A local file could be:
    ``file://localhost/path/

In [8]:
columns = ['a_0','a_1','cat_0','str1_0','str1_1']

**Option 1**: Load the entire dataset and then filter out the columns you're interested in

In [ ]:
%%memit 
df_filter = pd.read_parquet("timeseries.parquet")[columns]

peak memory: 3691.34 MiB, increment: 2100.61 MiB


In [11]:
%%time
df_filter = pd.read_parquet("timeseries.parquet")[columns]
df_filter.head()

CPU times: user 1.61 s, sys: 786 ms, total: 2.4 s
Wall time: 567 ms


,a_0,a_1,cat_0,str1_0,str1_1
timestamp,,,,,
2020-01-01 00:00:00,0.703083,0.883547,cat2,BYYJ7Z960GUIPORHM9C302I747YG7DEL,H9MJGE9JFGEQ27TZ88LDXI3F39HHPB7U
2020-01-01 00:01:00,0.709437,0.849775,cat3,VPAKHHCHCF1GNQY989CGBEJWS2K6ZLYH,ROHA1BV2ZG9OPIEGOMRZKRBNA6W67L93
2020-01-01 00:02:00,0.283528,0.758266,cat2,HYJS2ZMEZ5HVSJVKF25P3DM4FZCNGC1O,MTX4LRBHB8ON2QDMS5Y0R2W3E6JZQ1IN
2020-01-01 00:03:00,0.974066,0.668060,cat4,U11DHD90WA52AATBY62IZSU0YVCVA74W,9P741U1IH582707MI2IAHDLU7XN6SUHI
2020-01-01 00:04:00,0.633060,0.337518,cat3,J9R9NDOFM9F12RZD4C0WE2DTBHMDMANI,CYZIE06HTNSTCPGMDOKJ4XBV4XGJ3ZZI


**Option 2**: Read only the columns you're interested in. 

In [ ]:
%%memit 
df_filter = pd.read_parquet("timeseries.parquet",columns=columns)

peak memory: 2820.34 MiB, increment: 0.00 MiB


In [14]:
%%time
df_filter = pd.read_parquet("timeseries.parquet",columns=columns)
df_filter.head()

CPU times: user 235 ms, sys: 148 ms, total: 383 ms
Wall time: 141 ms


,a_0,a_1,cat_0,str1_0,str1_1
timestamp,,,,,
2020-01-01 00:00:00,0.703083,0.883547,cat2,BYYJ7Z960GUIPORHM9C302I747YG7DEL,H9MJGE9JFGEQ27TZ88LDXI3F39HHPB7U
2020-01-01 00:01:00,0.709437,0.849775,cat3,VPAKHHCHCF1GNQY989CGBEJWS2K6ZLYH,ROHA1BV2ZG9OPIEGOMRZKRBNA6W67L93
2020-01-01 00:02:00,0.283528,0.758266,cat2,HYJS2ZMEZ5HVSJVKF25P3DM4FZCNGC1O,MTX4LRBHB8ON2QDMS5Y0R2W3E6JZQ1IN
2020-01-01 00:03:00,0.974066,0.668060,cat4,U11DHD90WA52AATBY62IZSU0YVCVA74W,9P741U1IH582707MI2IAHDLU7XN6SUHI
2020-01-01 00:04:00,0.633060,0.337518,cat3,J9R9NDOFM9F12RZD4C0WE2DTBHMDMANI,CYZIE06HTNSTCPGMDOKJ4XBV4XGJ3ZZI


You can use the magic command `%time` and `%memit` to compare the time and the memory usage of the two calls.

Not all the reading methods in Pandas has an option to read a subset of columns.

## Use efficient datatypes

The default pandas data types are not the most memory efficient. This is especially true for text data columns with relatively few unique values (commonly referred to as “low-cardinality” data). 

Using more efficient data types reduces the memory size of a dataframe, so you can store larger datasets in memory.

In [15]:
df = pd.read_parquet("timeseries.parquet",columns=['a_0','b_0','c_0','cat_0','str1_0','str2_0'])

Look at the data types of each column

In [16]:
df.dtypes

a_0       float64
b_0       float64
c_0         int64
cat_0         str
str1_0        str
str2_0        str
dtype: object

Look at the memory usage of the dataframe. The `memory_usage()` method returns the memory usage of each column in bytes.

In [17]:
df.memory_usage(deep=True)

Index     16819200
a_0       16819200
b_0       16819200
c_0       16819200
cat_0     25228800
str1_0    84096000
str2_0    84096000
dtype: int64

Compute the size of the dataframe. You should get the same result with the `info(memory_usage='deep')` method.

In [19]:
mem = df.memory_usage(deep=True)
mem.sum()/1024/1024

np.float64(248.62060546875)

In [20]:
df.info(memory_usage='deep')

<class 'pandas.DataFrame'>
DatetimeIndex: 2102400 entries, 2020-01-01 00:00:00 to 2023-12-30 23:59:00
Data columns (total 6 columns):
 #   Column  Dtype  
---  ------  -----  
 0   a_0     float64
 1   b_0     float64
 2   c_0     int64  
 3   cat_0   str    
 4   str1_0  str    
 5   str2_0  str    
dtypes: float64(2), int64(1), str(3)
memory usage: 248.6 MB


The result of `memory_usage` show that the columns taking up much more memory are 'str1_0','str2_0','cat_0'. It seems normal for 'str1_0','str2_0' columns because those columns contains random strings. But 'cat_0' column has just a few unique values, so it’s a good candidate for converting to a `pandas.Categorical`. 

With a `pandas.Categorical`, we store each unique name once and use space-efficient integers to know which specific name is used in each row.

First, we copy our dataframe to a new one.

In [21]:
df2 = df.copy()

Try to change to column type to Pandas.category using the `astype()` method

In [23]:
df2["cat_0"] = df2["cat_0"].astype("category")

Check with dtypes that the column type has changed

In [25]:
df2.dtypes

a_0        float64
b_0        float64
c_0          int64
cat_0     category
str1_0         str
str2_0         str
dtype: object

Compute the memory usage of each column for this new dataframe.

In [27]:
df2.memory_usage(deep=True)

Index     16819200
a_0       16819200
b_0       16819200
c_0       16819200
cat_0      2102461
str1_0    84096000
str2_0    84096000
dtype: int64

We can go a bit further and downcast the numeric columns to their smallest types using pandas.to_numeric(). The "c_0" column contains number between 0 and 100. So it can be downcast to unsigned. If float precision is sufficient for columns 'a_0' et 'b_0', it is also possible to downcast to float. Be careful when you downcast, you lose precision and so you can propagate error during the processing.

In [29]:
df2["c_0"] = pd.to_numeric(df2["c_0"], downcast="unsigned")
df2[["a_0", "b_0"]] = df2[["a_0", "b_0"]].apply(pd.to_numeric, downcast="float")

Check the types and the memory usage of the columns

In [31]:
df2.dtypes

a_0        float32
b_0        float32
c_0          uint8
cat_0     category
str1_0         str
str2_0         str
dtype: object

In [32]:
df2.memory_usage(deep=True)

Index     16819200
a_0        8409600
b_0        8409600
c_0        2102400
cat_0      2102461
str1_0    84096000
str2_0    84096000
dtype: int64

Compute the memory reduction

In [34]:
reduction = df2.memory_usage(deep=True).sum() / df.memory_usage(deep=True).sum()
print(f"{reduction:0.2f}")

0.79


# Use chunking

Some problem are embarrasingly parallel and so can be processed with chunking, which means by splitting a large problem into a bunch of small problems. 
For example, converting an big file into several smaller files and repeating the processing for each file in a directory. 
As long as each chunk fits in memory, you can work with datasets that are much larger than memory.

In [35]:
N = 12
starts = [f"20{i:>02d}-01-01" for i in range(N)]
ends = [f"20{i:>02d}-12-31" for i in range(N)]
pathlib.Path("data/timeseries").mkdir(parents=True,exist_ok=True)
for i, (start, end) in enumerate(zip(starts, ends)):
    ts = make_timeseries(start=start, end=end, freq="1min", seed=i)
    ts.to_parquet(f"data/timeseries/ts-{i:0>2d}.parquet")

Count the occurence of the values in the "c" column for all the files.

In [37]:
%%time

files = pathlib.Path("data/timeseries/").glob("ts*.parquet")
counts = pd.Series(dtype=int)

for path in files:
    df = pd.read_parquet(path)
    counts = counts.add(df["c"].value_counts(), fill_value=0)

counts.astype(int)

CPU times: user 804 ms, sys: 223 ms, total: 1.03 s
Wall time: 475 ms


c
1     63717
2     63625
3     63634
4     63404
5     63906
      ...  
95    63398
96    63527
97    63951
98    63290
99    63255
Length: 99, dtype: int64

Some readers, like pandas.read_csv(), offer parameters to control the chunksize when reading a single file. 
In that case, it is possible to read a file chunk by chunk in order to process it.

In [38]:
df = make_timeseries(start="2023-01-01", end="2023-12-31", freq="1min", seed=10)
df.to_csv("data/timeseries.csv")

Try to count the occurence of the values in the "c" column for the CSV file by process it chunk by chunk. You need to use the parameter `chunksize` in the `read_csv`method. 

In [40]:
counts = pd.Series(dtype=int)
with pd.read_csv("data/timeseries.csv",chunksize=1000) as reader:
    for chunk in reader:
        counts = counts.add(chunk["c"].value_counts(), fill_value=0)

counts.astype(int)

c
1     5260
2     5246
3     5227
4     5280
5     5321
      ... 
95    5330
96    5473
97    5431
98    5240
99    5304
Length: 99, dtype: int64

In [41]:
%%memit
counts = pd.Series(dtype=int)
with pd.read_csv("data/timeseries.csv",chunksize=1000) as reader:
    for chunk in reader:
        counts = counts.add(chunk["c"].value_counts(), fill_value=0)

counts.astype(int)

peak memory: 2699.34 MiB, increment: 0.00 MiB


In [42]:
%%memit
df = pd.read_csv("data/timeseries.csv")
df["c"].value_counts().astype(int)

peak memory: 2835.05 MiB, increment: 135.70 MiB
